# ACLED Africa: spatiotemporal grid model for protest diffusion

**Research question:** How do protests spread over time and space in Africa?

This notebook transforms ACLED protest events into a **grid-cell x month**
panel using an independent African land grid. Grid cells are created from Natural
Earth country boundaries before protest outcomes are examined, so cells with no
recorded protests remain valid zero observations.

Main design:
- Equal-area **50 x 50 km African land grid**
- Robustness checks at **25, 50, and 100 km**
- Predictor month `t` forecasts protest occurrence in month `t+1`
- 2000-2012 outcomes: training
- 2013-2015 outcomes: validation and model selection
- 2016-2020 outcomes: final held-out test

Hypotheses:
- **H1 Spatial diffusion:** protests in nearby cells predict protest risk in the
  focal cell next month after accounting for the focal cell's history.
- **H2 Scale robustness:** the spatial pattern remains broadly similar across
  25, 50, and 100 km grids.
- **H3 Fatality heterogeneity:** nonfatal neighboring protests have a stronger
  predictive association than fatal neighboring protests.

Boundary source: Natural Earth Admin-0 Countries, public-domain map data:
https://www.naturalearthdata.com/downloads/10m-cultural-vectors/10m-admin-0-countries/


## 0. Installation

Run the cell below only if your VSCode kernel is missing the required packages.


In [ ]:
# Run this cell in Colab if packages are missing:
# %pip install -q geopandas pyogrio shapely statsmodels scikit-learn seaborn


In [ ]:
from pathlib import Path
import math
import sys
import subprocess
import urllib.request
import warnings
warnings.filterwarnings("default")

import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import box
import statsmodels.api as sm

try:
    from IPython.display import display, Image
except Exception:
    Image = None
    def display(x):
        print(x)

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    precision_recall_fscore_support,
    precision_recall_curve,
)
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

REPO_URL = "https://github.com/Mat99999/acled-spatiotemporal-conflict-model.git"
PROJECT_FOLDER_NAME = "acled_spatiotemporal_project"
DATA_SUBDIR = Path("data") / "new data"
DATA_FILENAME = "comprehensive protest data.csv"

NATURAL_EARTH_URL = (
    "https://naturalearth.s3.amazonaws.com/10m_cultural/"
    "ne_10m_admin_0_countries.zip"
)
# Africa Albers Equal Area Conic. Distances are measured in meters.
AFRICA_EQUAL_AREA_CRS = (
    "+proj=aea +lat_1=-18 +lat_2=21 +lat_0=0 +lon_0=25 "
    "+datum=WGS84 +units=m +no_defs"
)
MIN_LAND_AREA_KM2 = 1.0


def running_in_colab():
    return "google.colab" in sys.modules


def find_project_dir():
    cwd = Path.cwd().resolve()
    candidates = [
        cwd,
        cwd.parent,
        cwd / PROJECT_FOLDER_NAME,
        cwd.parent / PROJECT_FOLDER_NAME,
    ]
    for candidate in candidates:
        data_path = candidate / DATA_SUBDIR / DATA_FILENAME
        if data_path.exists():
            return candidate

    if running_in_colab() and REPO_URL:
        clone_dir = Path("/content") / PROJECT_FOLDER_NAME
        if not clone_dir.exists():
            subprocess.run(["git", "clone", REPO_URL, str(clone_dir)], check=True)
        return clone_dir

    raise FileNotFoundError(
        "Could not locate the project folder. Expected this file under "
        f"{DATA_SUBDIR}: {DATA_FILENAME}."
    )


PROJECT_DIR = find_project_dir()
DATA_DIR = PROJECT_DIR / DATA_SUBDIR
DATA_PATH = DATA_DIR / DATA_FILENAME
REFERENCE_DIR = PROJECT_DIR / "data" / "reference"
OUTPUT_DIR = PROJECT_DIR / "outputs"
TABLES_DIR = OUTPUT_DIR / "tables"
FIGURES_DIR = OUTPUT_DIR / "figures"
PANELS_DIR = OUTPUT_DIR / "panels"
for directory in [REFERENCE_DIR, TABLES_DIR, FIGURES_DIR, PANELS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)


def save_and_show(fig, filename, dpi=140):
    path = FIGURES_DIR / filename
    fig.savefig(path, dpi=dpi, bbox_inches="tight")
    if Image is not None:
        display(Image(filename=str(path)))
    else:
        print(f"Saved figure: {path}")
    plt.close(fig)


RANDOM_STATE = 42
PRIMARY_GRID_KM = 50
ROBUSTNESS_GRID_KM = [25, 50, 100]
RUN_ROBUSTNESS = True  # Runs 25, 50, and 100 km validation robustness checks.

ANALYSIS_START = pd.Timestamp("2000-01-01")
ANALYSIS_END = pd.Timestamp("2020-12-31")
TRAIN_TARGET_END = pd.Timestamp("2012-12-01")
VALID_TARGET_START = pd.Timestamp("2013-01-01")
VALID_TARGET_END = pd.Timestamp("2015-12-01")
TEST_TARGET_START = pd.Timestamp("2016-01-01")
TEST_TARGET_END = pd.Timestamp("2020-12-01")

print("Data:", DATA_PATH)
print("Project:", PROJECT_DIR)
print("Output:", OUTPUT_DIR)


## 1. Load and validate the comprehensive ACLED protest data

The source is one protest-only ACLED CSV covering 2000-2020. The notebook creates
training, validation, and test periods later using the **month being predicted**.

Place `comprehensive protest data.csv` under `data/new data/` in the project.
When running interactively in Colab, the next cell also permits direct upload if
the file is not present in that folder.


In [ ]:
if not DATA_PATH.exists() and running_in_colab():
    from google.colab import files

    print(f"Upload: {DATA_FILENAME}")
    uploaded = files.upload()
    if DATA_FILENAME not in uploaded:
        raise FileNotFoundError(
            f"Expected an uploaded file named exactly: {DATA_FILENAME}"
        )
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    DATA_PATH.write_bytes(uploaded[DATA_FILENAME])

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Data file not found: {DATA_PATH}\n"
        f"Place {DATA_FILENAME} under {DATA_SUBDIR}."
    )

df = pd.read_csv(DATA_PATH, encoding="utf-8-sig")
df.columns = [c.strip().lower() for c in df.columns]

df["event_date"] = pd.to_datetime(df["event_date"], errors="coerce")
df["year"] = pd.to_numeric(df["year"], errors="coerce").astype("Int64")
df["fatalities"] = pd.to_numeric(
    df["fatalities"], errors="coerce"
).fillna(0)
df["fatal_protest"] = (df["fatalities"] >= 1).astype(int)
df["latitude"] = pd.to_numeric(df["latitude"], errors="coerce")
df["longitude"] = pd.to_numeric(df["longitude"], errors="coerce")
df["month"] = df["event_date"].dt.to_period("M").dt.to_timestamp()

# Protect against accidentally supplying the full, unfiltered ACLED dataset.
if "event_type" in df.columns:
    non_protests = (df["event_type"] != "Protests").sum()
    if non_protests:
        print(f"Filtering out {non_protests:,} non-protest records.")
        df = df[df["event_type"] == "Protests"].copy()

required = [
    "event_id_cnty",
    "event_date",
    "month",
    "latitude",
    "longitude",
    "country",
    "event_type",
    "sub_event_type",
    "fatalities",
    "fatal_protest",
]
missing_columns = [c for c in required if c not in df.columns]
if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

print("Rows:", len(df))
print("Date range:", df["event_date"].min(), "to", df["event_date"].max())
print("Event types:", df["event_type"].dropna().unique())
display(df[required].isna().sum().to_frame("missing"))
display(df.head(3))


In [ ]:
summary = pd.DataFrame(
    {
        "metric": [
            "protest_events",
            "countries",
            "months",
            "fatalities_total",
            "fatal_protest_events",
            "share_nonfatal_protests",
        ],
        "value": [
            len(df),
            df["country"].nunique(),
            df["month"].nunique(),
            int(df["fatalities"].sum()),
            int(df["fatal_protest"].sum()),
            round((df["fatal_protest"] == 0).mean(), 3),
        ],
    }
)
by_type = (
    df["event_type"].value_counts().rename_axis("event_type").reset_index(name="events")
)
by_subtype = (
    df["sub_event_type"]
    .value_counts()
    .rename_axis("sub_event_type")
    .reset_index(name="events")
)
by_period = pd.DataFrame(
    {
        "period": ["training", "validation", "final_test"],
        "start": ["2000-01", "2013-01", "2016-01"],
        "end": ["2012-12", "2015-12", "2020-12"],
    }
)

display(summary)
display(by_type)
display(by_subtype)
display(by_period)
summary.to_csv(TABLES_DIR / "01_data_summary.csv", index=False)
by_type.to_csv(TABLES_DIR / "02_event_types.csv", index=False)
by_subtype.to_csv(TABLES_DIR / "02b_protest_sub_event_types.csv", index=False)
by_period.to_csv(TABLES_DIR / "02c_analysis_periods.csv", index=False)

monthly = df.groupby("month").size().rename("events").reset_index()
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(monthly["month"], monthly["events"], color="#2F5D8C", linewidth=1.8)
ax.axvline(VALID_TARGET_START, color="#C8893D", linestyle="--", label="Validation")
ax.axvline(TEST_TARGET_START, color="#B04A3A", linestyle="--", label="Final test")
ax.set_title("ACLED protest events per month, Africa 2000-2020")
ax.set_xlabel("Month")
ax.set_ylabel("Protest events")
ax.legend()
fig.tight_layout()
save_and_show(fig, "events_per_month.png", dpi=160)


## 2. Construct an independent African land grid

The grid is created from Natural Earth country boundaries, not from ACLED event
locations. This keeps eligible African land cells even when they never experience
a recorded protest. The boundary is projected to an Africa-wide equal-area
coordinate system before square kilometer cells are generated.

Cells containing less than 1 square kilometer of land are removed to avoid cells
that only touch a coastline or border at a single point.


In [ ]:
def load_africa_boundary():
    zip_path = REFERENCE_DIR / "ne_10m_admin_0_countries.zip"
    if not zip_path.exists():
        print("Downloading Natural Earth country boundaries...")
        urllib.request.urlretrieve(NATURAL_EARTH_URL, zip_path)

    world = gpd.read_file(f"zip://{zip_path}")
    continent_col = "CONTINENT" if "CONTINENT" in world.columns else "continent"
    africa = world[world[continent_col] == "Africa"].copy()
    if africa.empty:
        raise ValueError("No African country polygons were found.")
    return africa.to_crs(AFRICA_EQUAL_AREA_CRS)


AFRICA_COUNTRIES = load_africa_boundary()
AFRICA_LAND = AFRICA_COUNTRIES.dissolve().geometry.iloc[0]
GRID_CACHE = {}


def build_africa_land_grid(grid_km):
    if grid_km in GRID_CACHE:
        return GRID_CACHE[grid_km].copy()

    cell_size = grid_km * 1000.0
    minx, miny, maxx, maxy = AFRICA_LAND.bounds
    gx_min = math.floor(minx / cell_size)
    gx_max = math.ceil(maxx / cell_size)
    gy_min = math.floor(miny / cell_size)
    gy_max = math.ceil(maxy / cell_size)

    records = []
    for gx in range(gx_min, gx_max):
        for gy in range(gy_min, gy_max):
            geom = box(
                gx * cell_size,
                gy * cell_size,
                (gx + 1) * cell_size,
                (gy + 1) * cell_size,
            )
            if not geom.intersects(AFRICA_LAND):
                continue
            land_area_km2 = geom.intersection(AFRICA_LAND).area / 1_000_000
            if land_area_km2 < MIN_LAND_AREA_KM2:
                continue
            records.append(
                {
                    "cell_id": f"{grid_km}km_{gx}_{gy}",
                    "grid_x": gx,
                    "grid_y": gy,
                    "grid_km": grid_km,
                    "land_area_km2": land_area_km2,
                    "geometry": geom,
                }
            )

    grid = gpd.GeoDataFrame(records, crs=AFRICA_EQUAL_AREA_CRS)
    GRID_CACHE[grid_km] = grid
    return grid.copy()


def assign_events_to_grid(events, grid):
    clean = events.dropna(subset=["latitude", "longitude", "month"]).copy()
    points = gpd.GeoDataFrame(
        clean,
        geometry=gpd.points_from_xy(clean["longitude"], clean["latitude"]),
        crs="EPSG:4326",
    ).to_crs(AFRICA_EQUAL_AREA_CRS)

    cell_size = float(grid["grid_km"].iloc[0]) * 1000.0
    points["grid_x"] = np.floor(points.geometry.x / cell_size).astype(int)
    points["grid_y"] = np.floor(points.geometry.y / cell_size).astype(int)
    points["cell_id"] = (
        str(int(grid["grid_km"].iloc[0]))
        + "km_"
        + points["grid_x"].astype(str)
        + "_"
        + points["grid_y"].astype(str)
    )

    valid = grid[["cell_id", "grid_x", "grid_y"]]
    assigned = points.drop(columns="geometry").merge(
        valid, on=["cell_id", "grid_x", "grid_y"], how="inner"
    )
    dropped = len(points) - len(assigned)
    if dropped:
        print(f"Warning: {dropped} events fell outside retained African land cells.")
    return assigned


land_grid100 = build_africa_land_grid(PRIMARY_GRID_KM)
df_grid = assign_events_to_grid(df, land_grid100)
print("Eligible African land cells:", len(land_grid100))
print("Cells with at least one recorded protest:", df_grid["cell_id"].nunique())
print("Cells with no recorded protests:", len(land_grid100) - df_grid["cell_id"].nunique())

fig, ax = plt.subplots(figsize=(8, 8))
land_grid100.boundary.plot(ax=ax, linewidth=0.12, color="#777777")
AFRICA_COUNTRIES.boundary.plot(ax=ax, linewidth=0.35, color="#222222")
ax.set_title("Independent 100 km African land grid")
ax.set_axis_off()
fig.tight_layout()
save_and_show(fig, "africa_land_grid_50km.png", dpi=180)


## 3. Build the complete cell-month panel and lagged features

Every retained land cell receives one row for every month. Months without a
recorded protest are coded as zero.

Each row represents predictor month `t`. Local and neighboring protest measures
from month `t` are used to predict protest occurrence in month `t+1`. The
`target_month` field records the month being predicted and is used for all
training, validation, and testing splits.


In [ ]:
def safe_name(value):
    return (
        str(value)
        .lower()
        .replace(" ", "_")
        .replace("/", "_")
        .replace("-", "_")
    )


def build_panel(events, grid_km, include_sub_event_types=True):
    grid = build_africa_land_grid(grid_km)
    e = assign_events_to_grid(events, grid)
    e = e[
        (e["event_date"] >= ANALYSIS_START)
        & (e["event_date"] <= ANALYSIS_END)
    ].copy()
    e["fatal_protest"] = (
        pd.to_numeric(e["fatal_protest"], errors="coerce").fillna(0) >= 1
    ).astype(int)
    e["nonfatal_protest"] = (e["fatal_protest"] == 0).astype(int)

    agg = (
        e.groupby(["cell_id", "grid_x", "grid_y", "month"])
        .agg(
            events=("event_id_cnty", "count"),
            fatalities=("fatalities", "sum"),
            fatal_protests=("fatal_protest", "sum"),
            nonfatal_protests=("nonfatal_protest", "sum"),
        )
        .reset_index()
    )

    subtype_cols = []
    if include_sub_event_types:
        subtype_counts = (
            e.pivot_table(
                index=["cell_id", "month"],
                columns="sub_event_type",
                values="event_id_cnty",
                aggfunc="count",
                fill_value=0,
            )
            .reset_index()
        )
        subtype_counts.columns = ["cell_id", "month"] + [
            f"subtype_{safe_name(c)}" for c in subtype_counts.columns[2:]
        ]
        subtype_cols = [c for c in subtype_counts.columns if c.startswith("subtype_")]
        agg = agg.merge(subtype_counts, on=["cell_id", "month"], how="left")

    cells = grid[["cell_id", "grid_x", "grid_y", "land_area_km2"]].copy()
    months = pd.date_range(ANALYSIS_START, ANALYSIS_END, freq="MS")
    panel = cells.merge(pd.DataFrame({"month": months}), how="cross")
    panel = panel.merge(
        agg, on=["cell_id", "grid_x", "grid_y", "month"], how="left"
    )

    count_cols = [
        "events",
        "fatalities",
        "fatal_protests",
        "nonfatal_protests",
    ] + subtype_cols
    panel[count_cols] = panel[count_cols].fillna(0)
    panel["protest"] = (panel["events"] > 0).astype(int)
    panel = panel.sort_values(["cell_id", "month"]).reset_index(drop=True)

    # Target-relative local history: lag 1 is predictor month t for target t+1.
    for target_lag in [1, 2, 3, 6, 12]:
        shift_n = target_lag - 1
        panel[f"local_events_lag_{target_lag}"] = (
            panel.groupby("cell_id")["events"].shift(shift_n).fillna(0)
        )
        panel[f"local_protest_lag_{target_lag}"] = (
            panel.groupby("cell_id")["protest"].shift(shift_n).fillna(0)
        )
        panel[f"local_fatalities_lag_{target_lag}"] = (
            panel.groupby("cell_id")["fatalities"].shift(shift_n).fillna(0)
        )
        panel[f"local_fatal_protests_lag_{target_lag}"] = (
            panel.groupby("cell_id")["fatal_protests"].shift(shift_n).fillna(0)
        )
        panel[f"local_nonfatal_protests_lag_{target_lag}"] = (
            panel.groupby("cell_id")["nonfatal_protests"].shift(shift_n).fillna(0)
        )

    panel["local_events_roll_6"] = (
        panel.groupby("cell_id")["events"]
        .transform(lambda s: s.rolling(6, min_periods=1).sum())
    )
    panel["local_events_roll_12"] = (
        panel.groupby("cell_id")["events"]
        .transform(lambda s: s.rolling(12, min_periods=1).sum())
    )

    # Neighbor features use protests in predictor month t, without an extra shift.
    base = panel[
        [
            "grid_x",
            "grid_y",
            "month",
            "events",
            "fatalities",
            "protest",
            "fatal_protests",
            "nonfatal_protests",
        ]
    ].copy()
    shifted_neighbors = []
    for dx in [-1, 0, 1]:
        for dy in [-1, 0, 1]:
            if dx == 0 and dy == 0:
                continue
            tmp = base.copy()
            tmp["grid_x"] = tmp["grid_x"] - dx
            tmp["grid_y"] = tmp["grid_y"] - dy
            shifted_neighbors.append(tmp)

    neigh = pd.concat(shifted_neighbors, ignore_index=True)
    neigh_agg = (
        neigh.groupby(["grid_x", "grid_y", "month"])
        .agg(
            neighbor_protests=("events", "sum"),
            neighbor_fatalities=("fatalities", "sum"),
            neighbor_protest_cells=("protest", "sum"),
            neighbor_fatal_protests=("fatal_protests", "sum"),
            neighbor_nonfatal_protests=("nonfatal_protests", "sum"),
        )
        .reset_index()
    )
    panel = panel.merge(
        neigh_agg, on=["grid_x", "grid_y", "month"], how="left"
    )
    neighbor_cols = [
        "neighbor_protests",
        "neighbor_fatalities",
        "neighbor_protest_cells",
        "neighbor_fatal_protests",
        "neighbor_nonfatal_protests",
    ]
    panel[neighbor_cols] = panel[neighbor_cols].fillna(0)

    panel["target_protest_next_month"] = (
        panel.groupby("cell_id")["protest"].shift(-1)
    )
    panel["target_events_next_month"] = (
        panel.groupby("cell_id")["events"].shift(-1)
    )
    panel["target_month"] = panel["month"] + pd.offsets.MonthBegin(1)
    panel["month_num"] = panel["target_month"].dt.month
    panel["year"] = panel["target_month"].dt.year
    panel["sin_month"] = np.sin(2 * np.pi * panel["month_num"] / 12)
    panel["cos_month"] = np.cos(2 * np.pi * panel["month_num"] / 12)

    return panel.dropna(subset=["target_protest_next_month"]).reset_index(drop=True)


panel_main = build_panel(df, PRIMARY_GRID_KM)
print(panel_main.shape)
print("Cells:", panel_main["cell_id"].nunique())
print(
    "Positive target share:",
    round(panel_main["target_protest_next_month"].mean(), 4),
)
display(panel_main.head())
panel_main.to_csv(PANELS_DIR / "panel_50km_month.csv", index=False)


In [ ]:
panel_summary = pd.DataFrame(
    {
        "metric": [
            "grid_km",
            "cell_months",
            "cells",
            "months",
            "positive_target_share",
            "mean_events_when_positive",
        ],
        "value": [
            PRIMARY_GRID_KM,
            len(panel_main),
            panel_main["cell_id"].nunique(),
            panel_main["month"].nunique(),
            round(panel_main["target_protest_next_month"].mean(), 4),
            round(panel_main.loc[panel_main["events"] > 0, "events"].mean(), 2),
        ],
    }
)
display(panel_summary)
panel_summary.to_csv(TABLES_DIR / "03_panel_50km_summary.csv", index=False)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
panel_main.groupby("month")["protest"].mean().plot(
    ax=axes[0], color="#2F5D8C"
)
axes[0].axvline(
    VALID_TARGET_START, color="#C8893D", linestyle="--", linewidth=1.2
)
axes[0].axvline(
    TEST_TARGET_START, color="#B04A3A", linestyle="--", linewidth=1.2
)
axes[0].set_title("Share of active protest cells per month")
axes[0].set_ylabel("Share with >=1 protest")
panel_main.groupby("cell_id")["events"].sum().clip(upper=100).hist(
    ax=axes[1], bins=30, color="#C8893D"
)
axes[1].set_title("Distribution of total cell protests, clipped at 100")
axes[1].set_xlabel("Protests per cell, 2000-2020")
fig.tight_layout()
save_and_show(fig, "panel_diagnostics_50km.png", dpi=160)


## 4. Modeling strategy

We compare pre-specified feature blocks:

- **A_place_time:** target year, season, and geographic position
- **B_local_history:** A plus the focal cell's protest history
- **C_local_plus_neighbors:** B plus total protests and active protest cells among
  the eight neighboring cells in predictor month `t`
- **D_fatality_split:** local history plus neighboring active cells and separate
  fatal/nonfatal neighboring protest counts
- **E_extended:** D plus local fatality history and protest subtype variables

H1 is assessed by comparing C against B. H3 uses D, which deliberately excludes
`neighbor_protests` because total protests equal fatal plus nonfatal protests.

Model choices and thresholds are developed on 2013-2015. The 2016-2020 period is
evaluated only after those choices are established.


In [ ]:
BASE_FEATURES = ["grid_x", "grid_y", "year", "sin_month", "cos_month"]
LOCAL_FEATURES = BASE_FEATURES + [
    "local_events_lag_1",
    "local_events_lag_2",
    "local_events_lag_3",
    "local_events_lag_6",
    "local_events_lag_12",
    "local_protest_lag_1",
    "local_protest_lag_2",
    "local_protest_lag_3",
    "local_protest_lag_6",
    "local_protest_lag_12",
    "local_events_roll_6",
    "local_events_roll_12",
]
SPATIAL_FEATURES = LOCAL_FEATURES + [
    "neighbor_protests",
    "neighbor_protest_cells",
]
# Do not include neighbor_protests here: it is exactly fatal + nonfatal.
FATALITY_SPLIT_FEATURES = LOCAL_FEATURES + [
    "neighbor_protest_cells",
    "neighbor_nonfatal_protests",
    "neighbor_fatal_protests",
]
SUBTYPE_FEATURES = [c for c in panel_main.columns if c.startswith("subtype_")]
EXTENDED_FEATURES = FATALITY_SPLIT_FEATURES + [
    "neighbor_fatalities",
    "local_fatalities_lag_1",
    "local_fatalities_lag_6",
    "local_fatalities_lag_12",
    "local_fatal_protests_lag_1",
    "local_fatal_protests_lag_6",
    "local_fatal_protests_lag_12",
    "local_nonfatal_protests_lag_1",
    "local_nonfatal_protests_lag_6",
    "local_nonfatal_protests_lag_12",
] + SUBTYPE_FEATURES

FEATURE_SETS = {
    "A_place_time": BASE_FEATURES,
    "B_local_history": LOCAL_FEATURES,
    "C_local_plus_neighbors": SPATIAL_FEATURES,
    "D_fatality_split": FATALITY_SPLIT_FEATURES,
    "E_extended": EXTENDED_FEATURES,
}


def period_mask(panel, start, end):
    return (panel["target_month"] >= start) & (panel["target_month"] <= end)


train_mask = panel_main["target_month"] <= TRAIN_TARGET_END
valid_mask = period_mask(panel_main, VALID_TARGET_START, VALID_TARGET_END)
test_mask = period_mask(panel_main, TEST_TARGET_START, TEST_TARGET_END)

print("Train rows:", train_mask.sum())
print("Validation rows:", valid_mask.sum())
print("Test rows:", test_mask.sum())
print(
    "Positive shares:",
    {
        "train": round(panel_main.loc[train_mask, "target_protest_next_month"].mean(), 4),
        "validation": round(panel_main.loc[valid_mask, "target_protest_next_month"].mean(), 4),
        "test": round(panel_main.loc[test_mask, "target_protest_next_month"].mean(), 4),
    },
)


In [ ]:
def evaluate_predictions(y_true, y_prob, threshold):
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob)
    y_pred = (y_prob >= threshold).astype(int)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="binary", zero_division=0
    )
    return {
        "roc_auc": roc_auc_score(y_true, y_prob),
        "avg_precision": average_precision_score(y_true, y_prob),
        "brier": brier_score_loss(y_true, y_prob),
        "threshold": threshold,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "positive_rate_pred": y_pred.mean(),
    }


def fit_models(panel, feature_sets, fit_end, eval_start, eval_end, thresholds=None):
    fit_mask = panel["target_month"] <= fit_end
    eval_mask = period_mask(panel, eval_start, eval_end)
    y_fit = panel.loc[fit_mask, "target_protest_next_month"].astype(int)
    y_eval = panel.loc[eval_mask, "target_protest_next_month"].astype(int)
    predictions = panel.loc[
        eval_mask,
        [
            "cell_id",
            "grid_x",
            "grid_y",
            "month",
            "target_month",
            "target_protest_next_month",
            "events",
            "neighbor_protests",
            "neighbor_nonfatal_protests",
            "neighbor_fatal_protests",
        ],
    ].copy()
    results = []

    for name, features in feature_sets.items():
        print(f"Fitting {name} on {len(panel):,} panel rows...", flush=True)
        features = [f for f in features if f in panel.columns]
        X_fit = panel.loc[fit_mask, features].fillna(0)
        X_eval = panel.loc[eval_mask, features].fillna(0)

        models = {
            "Logit": make_pipeline(
                StandardScaler(),
                LogisticRegression(
                    max_iter=1000,
                    class_weight="balanced",
                    random_state=RANDOM_STATE,
                ),
            ),
            "RF": RandomForestClassifier(
                n_estimators=100,
                min_samples_leaf=10,
                class_weight="balanced_subsample",
                n_jobs=-1,
                random_state=RANDOM_STATE,
            ),
        }

        for family, model in models.items():
            model.fit(X_fit, y_fit)
            probability = model.predict_proba(X_eval)[:, 1]
            model_name = f"{family}_{name}"
            threshold = (
                thresholds.get(model_name, 0.5)
                if thresholds is not None
                else 0.5
            )
            row = {
                "model": model_name,
                "feature_set": name,
                "n_features": len(features),
            }
            row.update(evaluate_predictions(y_eval, probability, threshold))
            results.append(row)
            predictions[f"p_{model_name}"] = probability

    results = pd.DataFrame(results).sort_values(
        ["avg_precision", "roc_auc"], ascending=False
    )
    return results, predictions


# Development evaluation: use this period for model and threshold choices.
validation_results_main, validation_pred_main = fit_models(
    panel_main,
    FEATURE_SETS,
    fit_end=TRAIN_TARGET_END,
    eval_start=VALID_TARGET_START,
    eval_end=VALID_TARGET_END,
)
display(validation_results_main)
validation_results_main.to_csv(
    TABLES_DIR / "04a_model_results_50km_validation.csv", index=False
)
validation_pred_main.to_csv(
    PANELS_DIR / "predictions_50km_validation.csv", index=False
)


def select_f1_thresholds(predictions):
    y = predictions["target_protest_next_month"].astype(int).to_numpy()
    selected = {}
    for col in [c for c in predictions.columns if c.startswith("p_")]:
        best_threshold = 0.5
        best_f1 = -1
        for threshold in np.linspace(0.01, 0.99, 99):
            pred = (predictions[col].to_numpy() >= threshold).astype(int)
            _, _, f1, _ = precision_recall_fscore_support(
                y, pred, average="binary", zero_division=0
            )
            if f1 > best_f1:
                best_f1 = f1
                best_threshold = threshold
        selected[col.removeprefix("p_")] = best_threshold
    return selected


VALIDATION_THRESHOLDS = select_f1_thresholds(validation_pred_main)

# Final evaluation: refit through 2015 and apply validation-selected thresholds.
results_main, pred_main = fit_models(
    panel_main,
    FEATURE_SETS,
    fit_end=VALID_TARGET_END,
    eval_start=TEST_TARGET_START,
    eval_end=TEST_TARGET_END,
    thresholds=VALIDATION_THRESHOLDS,
)
display(results_main)
results_main.to_csv(TABLES_DIR / "04b_model_results_50km_final_test.csv", index=False)
pred_main.to_csv(PANELS_DIR / "predictions_50km_final_test.csv", index=False)


## 5. Main interpretation: do neighboring cells add predictive information?


In [ ]:
comparison = results_main.copy()
comparison['model_family'] = comparison['model'].str.extract(r'^(Logit|RF)_')
pivot = comparison.pivot_table(index=['model_family','feature_set'], values=['roc_auc','avg_precision','brier','f1'], aggfunc='first').reset_index()
display(pivot)
pivot.to_csv(TABLES_DIR / '05_feature_set_comparison_50km.csv', index=False)

fig, ax = plt.subplots(figsize=(10,4.5))
plot_df = results_main.sort_values('avg_precision', ascending=True)
ax.barh(plot_df['model'], plot_df['avg_precision'], color='#2F5D8C')
ax.set_title('Test performance: Average Precision, 50 km grid')
ax.set_xlabel('Average precision')
ax.set_ylabel('')
fig.tight_layout()
save_and_show(fig, 'model_average_precision_50km.png', dpi=160)


## 6. Ranking, calibration, and formal H3 diagnostics

The classification thresholds below were selected using 2013-2015 validation
data and then applied unchanged to 2016-2020.

H3 is estimated using a linear-probability robustness model with standard errors
clustered by grid cell. This specification is used because fatal protests are
rare and the clustered logistic covariance can become numerically unstable. The
formal Wald test evaluates whether the nonfatal and fatal neighboring-protest
coefficients are equal. These remain associational, not causal, estimates.


In [ ]:
def precision_at_top_k(y_true, y_prob, top_shares=(0.01, 0.05, 0.10)):
    d = pd.DataFrame(
        {"y_true": np.asarray(y_true).astype(int), "y_prob": np.asarray(y_prob)}
    ).sort_values("y_prob", ascending=False)
    rows = []
    for share in top_shares:
        k = max(1, int(np.ceil(len(d) * share)))
        top = d.iloc[:k]
        rows.append(
            {
                "top_share": share,
                "n_flagged": k,
                "precision_at_top": top["y_true"].mean(),
                "recall_at_top": top["y_true"].sum() / d["y_true"].sum(),
                "lift_vs_base_rate": top["y_true"].mean() / d["y_true"].mean(),
            }
        )
    return pd.DataFrame(rows)


y_test = pred_main["target_protest_next_month"].astype(int)
model_cols = {
    "RF_B_local_history": "p_RF_B_local_history",
    "RF_C_local_plus_neighbors": "p_RF_C_local_plus_neighbors",
    "RF_D_fatality_split": "p_RF_D_fatality_split",
    "Logit_C_local_plus_neighbors": "p_Logit_C_local_plus_neighbors",
    "Logit_D_fatality_split": "p_Logit_D_fatality_split",
}
model_cols = {k: v for k, v in model_cols.items() if v in pred_main.columns}

threshold_rows = []
topk_rows = []
for model_name, col in model_cols.items():
    threshold = VALIDATION_THRESHOLDS[model_name]
    metrics = evaluate_predictions(y_test, pred_main[col], threshold)
    metrics["model"] = model_name
    metrics["base_rate"] = y_test.mean()
    threshold_rows.append(metrics)

    topk = precision_at_top_k(y_test, pred_main[col])
    topk.insert(0, "model", model_name)
    topk_rows.append(topk)

threshold_eval = pd.DataFrame(threshold_rows)[
    [
        "model",
        "base_rate",
        "threshold",
        "precision",
        "recall",
        "f1",
        "positive_rate_pred",
    ]
]
topk_eval = pd.concat(topk_rows, ignore_index=True)
display(threshold_eval)
display(topk_eval)
threshold_eval.to_csv(
    TABLES_DIR / "08_validation_selected_thresholds_final_test.csv", index=False
)
topk_eval.to_csv(TABLES_DIR / "09_precision_at_top_k_final_test.csv", index=False)

# H1 predictive uplift on the final test.
uplift_rows = []
for family in ["RF", "Logit"]:
    b = results_main[results_main["model"] == f"{family}_B_local_history"].iloc[0]
    c = results_main[
        results_main["model"] == f"{family}_C_local_plus_neighbors"
    ].iloc[0]
    uplift_rows.append(
        {
            "model_family": family,
            "delta_roc_auc_C_minus_B": c["roc_auc"] - b["roc_auc"],
            "delta_avg_precision_C_minus_B": (
                c["avg_precision"] - b["avg_precision"]
            ),
            "delta_brier_C_minus_B": c["brier"] - b["brier"],
        }
    )
neighbor_uplift = pd.DataFrame(uplift_rows)
display(neighbor_uplift)
neighbor_uplift.to_csv(
    TABLES_DIR / "10_neighbor_uplift_B_vs_C_final_test.csv", index=False
)

# H3 inference on pre-test data, with grid-cell clustered uncertainty.
h3_features = [f for f in FATALITY_SPLIT_FEATURES if f in panel_main.columns]
development_mask = panel_main["target_month"] <= VALID_TARGET_END
X_h3_raw = panel_main.loc[development_mask, h3_features].fillna(0)
y_h3 = panel_main.loc[
    development_mask, "target_protest_next_month"
].astype(int)
groups_h3 = panel_main.loc[development_mask, "cell_id"]

scaler_h3 = StandardScaler()
X_h3 = pd.DataFrame(
    scaler_h3.fit_transform(X_h3_raw),
    columns=h3_features,
    index=X_h3_raw.index,
)
X_h3 = sm.add_constant(X_h3, has_constant="add")
h3_fit = sm.OLS(y_h3, X_h3).fit(
    cov_type="cluster",
    cov_kwds={"groups": groups_h3},
)

h3_focus_names = [
    "neighbor_nonfatal_protests",
    "neighbor_fatal_protests",
]
h3_focus = pd.DataFrame(
    {
        "feature": h3_focus_names,
        "probability_change_per_1sd": h3_fit.params[h3_focus_names],
        "std_error_clustered": h3_fit.bse[h3_focus_names],
        "p_value": h3_fit.pvalues[h3_focus_names],
        "ci_low": h3_fit.conf_int().loc[h3_focus_names, 0],
        "ci_high": h3_fit.conf_int().loc[h3_focus_names, 1],
    }
)

restriction = np.zeros((1, len(h3_fit.params)))
restriction[0, list(h3_fit.params.index).index("neighbor_nonfatal_protests")] = 1
restriction[0, list(h3_fit.params.index).index("neighbor_fatal_protests")] = -1
h3_wald = h3_fit.wald_test(restriction, scalar=True)
h3_test = pd.DataFrame(
    {
        "hypothesis": [
            "H0: neighbor_nonfatal coefficient = neighbor_fatal coefficient"
        ],
        "wald_statistic": [float(h3_wald.statistic)],
        "p_value": [float(h3_wald.pvalue)],
    }
)

display(h3_focus)
display(h3_test)
h3_focus.to_csv(TABLES_DIR / "12a_h3_coefficients_50km.csv", index=False)
h3_test.to_csv(TABLES_DIR / "12b_h3_wald_test_50km.csv", index=False)


In [ ]:
# Fast plotting helpers: keep the analytical content, but avoid plotting tens of thousands of points.
def thin_curve(x, y, max_points=600):
    x = np.asarray(x)
    y = np.asarray(y)
    if len(x) <= max_points:
        return x, y
    idx = np.unique(np.linspace(0, len(x) - 1, max_points).astype(int))
    return x[idx], y[idx]


def fast_quantile_calibration(y_true, y_prob, n_bins=10):
    d = pd.DataFrame({'y_true': np.asarray(y_true).astype(int), 'y_prob': np.asarray(y_prob)})
    d = d.sort_values('y_prob').reset_index(drop=True)
    d['bin'] = pd.qcut(d.index, q=n_bins, labels=False, duplicates='drop')
    cal = d.groupby('bin', as_index=False).agg(
        mean_predicted_risk=('y_prob', 'mean'),
        observed_protest_rate=('y_true', 'mean'),
        n=('y_true', 'size')
    )
    return cal


base_rate = y_test.mean()

# Precision-recall curves.
fig, ax = plt.subplots(figsize=(8, 5))
ax.axhline(base_rate, color='#777777', linestyle='--', linewidth=1.2, label=f'Base rate ({base_rate:.3f})')
for model_name, col in model_cols.items():
    precision, recall, _ = precision_recall_curve(y_test, pred_main[col])
    recall_plot, precision_plot = thin_curve(recall, precision, max_points=600)
    ax.plot(recall_plot, precision_plot, linewidth=1.8, label=model_name)
ax.set_title('Precision-recall curves, test period')
ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.legend(fontsize=8)
fig.tight_layout()
save_and_show(fig, 'precision_recall_curves_50km.png', dpi=140)

# Calibration curves.
calibration_rows = []
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot([0, 1], [0, 1], color='#777777', linestyle='--', linewidth=1.2, label='Perfect calibration')
for model_name, col in model_cols.items():
    cal = fast_quantile_calibration(y_test, pred_main[col], n_bins=10)
    cal.insert(0, 'model', model_name)
    calibration_rows.append(cal)
    ax.plot(cal['mean_predicted_risk'], cal['observed_protest_rate'], marker='o', linewidth=1.6, label=model_name)
ax.set_title('Calibration curves, test period')
ax.set_xlabel('Mean predicted risk')
ax.set_ylabel('Observed protest rate')
ax.legend(fontsize=8)
fig.tight_layout()
save_and_show(fig, 'calibration_curves_50km.png', dpi=140)

calibration_table = pd.concat(calibration_rows, ignore_index=True)
calibration_table.to_csv(TABLES_DIR / '11_calibration_bins_50km.csv', index=False)

# Top-k precision plot using plain matplotlib rather than seaborn.
fig, ax = plt.subplots(figsize=(8, 5))
plot_topk = topk_eval.copy()
plot_topk['top_percent'] = (plot_topk['top_share'] * 100).astype(int).astype(str) + '%'
models = list(plot_topk['model'].unique())
top_levels = list(plot_topk['top_percent'].unique())
x = np.arange(len(top_levels))
width = 0.8 / max(1, len(models))
for j, model_name in enumerate(models):
    vals = plot_topk[plot_topk['model'] == model_name].set_index('top_percent').loc[top_levels, 'precision_at_top']
    ax.bar(x + (j - (len(models)-1)/2) * width, vals.values, width=width, label=model_name)
ax.axhline(base_rate, color='#777777', linestyle='--', linewidth=1.2, label='Base rate')
ax.set_title('Precision among highest-risk cell-months')
ax.set_xlabel('Flagged share of test cell-months')
ax.set_ylabel('Precision')
ax.set_xticks(x)
ax.set_xticklabels(top_levels)
ax.legend(fontsize=8)
fig.tight_layout()
save_and_show(fig, 'precision_at_top_k_50km.png', dpi=140)

print('Saved fast evaluation plots and calibration table.')


## 7. Map-like sanity check

This is not a formal map, but it helps assess whether observed and predicted protest risk form plausible geographic clusters.


In [ ]:
best_model_col = 'p_RF_D_fatality_split' if 'p_RF_D_fatality_split' in pred_main.columns else 'p_RF_C_local_plus_neighbors'
risk_map = pred_main.groupby(['grid_x','grid_y']).agg(
    observed_rate=('target_protest_next_month','mean'),
    predicted_risk=(best_model_col,'mean'),
    test_months=('month','nunique')
).reset_index()

fig, axes = plt.subplots(1,2,figsize=(12,5),sharex=True,sharey=True)
sc1 = axes[0].scatter(risk_map['grid_x'], risk_map['grid_y'], c=risk_map['observed_rate'], s=18, cmap='magma')
axes[0].set_title('Observed protest rate, test')
plt.colorbar(sc1, ax=axes[0], fraction=0.046)
sc2 = axes[1].scatter(risk_map['grid_x'], risk_map['grid_y'], c=risk_map['predicted_risk'], s=18, cmap='magma')
axes[1].set_title(f'Predicted risk: {best_model_col.replace("p_", "")}')
plt.colorbar(sc2, ax=axes[1], fraction=0.046)
for ax in axes:
    ax.set_xlabel('Grid x')
    ax.set_ylabel('Grid y')
fig.tight_layout()
save_and_show(fig, 'observed_vs_predicted_risk_map_50km.png', dpi=180)
risk_map.to_csv(PANELS_DIR / '06_test_risk_map_50km.csv', index=False)


## 8. Optional robustness check: 25, 50, and 100 km

This section uses the validation period, not the final test period, to compare
grid sizes. The 25 km panel is substantially larger than the main 50 km panel. The notebook
runs robustness by default here, but it may take longer and require more memory
than the main model.


In [ ]:
def run_grid_robustness(grid_sizes=(25, 50, 100)):
    rows = []
    for grid_km in grid_sizes:
        print(f"Running grid {grid_km} km...")
        panel = build_panel(df, grid_km)
        feature_sets = {
            "B_local_history": LOCAL_FEATURES,
            "C_local_plus_neighbors": SPATIAL_FEATURES,
            "D_fatality_split": FATALITY_SPLIT_FEATURES,
        }
        result, _ = fit_models(
            panel,
            feature_sets,
            fit_end=TRAIN_TARGET_END,
            eval_start=VALID_TARGET_START,
            eval_end=VALID_TARGET_END,
        )
        result["grid_km"] = grid_km
        result["cells"] = panel["cell_id"].nunique()
        result["cell_months"] = len(panel)
        validation_mask = period_mask(
            panel, VALID_TARGET_START, VALID_TARGET_END
        )
        result["positive_share_validation"] = panel.loc[
            validation_mask, "target_protest_next_month"
        ].mean()
        rows.append(result)
    return pd.concat(rows, ignore_index=True)


if RUN_ROBUSTNESS:
    robustness_results = run_grid_robustness(ROBUSTNESS_GRID_KM)
    display(robustness_results.sort_values(["grid_km", "model"]))
    robustness_results.to_csv(
        TABLES_DIR / "07_robustness_25_50_100km_validation.csv",
        index=False,
    )

    fig, ax = plt.subplots(figsize=(9, 4.5))
    temp = robustness_results.copy()
    temp["family"] = temp["model"].str.extract(r"^(Logit|RF)_")
    for (family, label), sub in temp.groupby(["family", "feature_set"]):
        sub = sub.sort_values("grid_km")
        ax.plot(
            sub["grid_km"],
            sub["avg_precision"],
            marker="o",
            label=f"{family}: {label}",
        )
    ax.set_title("Validation robustness across grid sizes")
    ax.set_xlabel("Grid size, km")
    ax.set_ylabel("Average precision")
    ax.legend(fontsize=8)
    fig.tight_layout()
    save_and_show(fig, "robustness_grid_sizes.png", dpi=160)
else:
    print("Robustness check skipped. Set RUN_ROBUSTNESS = True to run it.")
